
# DBSCAN Clustering
---

## 1. Introduction

Density-Based Spatial Clustering of Applications with Noise (DBSCAN) is an
unsupervised clustering algorithm that groups observations according to **local
density** instead of assigning points to centroid-based clusters.

Compared with methods such as K-Means, DBSCAN has several advantages:

- It does not require the number of clusters to be specified in advance  
- It can detect clusters with irregular shapes  
- It can label isolated observations as noise or outliers  
- It performs well when clusters are connected by density rather than by a central mean  

In this notebook, we apply DBSCAN using the **from-scratch implementation**
from the `rice_ml` package.

---



## 2. Mathematical Intuition Behind DBSCAN

DBSCAN is controlled by two main hyperparameters:

- **$\varepsilon$ (epsilon)**: the radius used to define a neighborhood  
- **`min_samples`**: the minimum number of nearby points required to declare a dense region  

### $\varepsilon$-Neighborhood

For a point $x$, the $\varepsilon$-neighborhood is

$$
N_\varepsilon(x) = \{ y \mid \text{distance}(x,y) \le \varepsilon \}.
$$

### Core Point

A point $x$ is called a **core point** if

$$
|N_\varepsilon(x)| \ge \text{min\_samples}.
$$

### Border Point and Noise Point

- A **border point** lies inside the neighborhood of a core point, but does not itself satisfy the density requirement  
- A **noise point** is not density-reachable from any core point  

DBSCAN builds each cluster by starting from a core point and repeatedly adding
all points that are density-reachable from it.



## 3. Online Dataset

For this example, we use the classic **Iris dataset**, loaded directly from a
public web URL. This means the notebook does **not** rely on any local CSV file.

The dataset contains flower measurements from three Iris species. Since DBSCAN
is distance-based, we focus on two continuous variables that are especially
useful for visual clustering:

- `Petal.Length`
- `Petal.Width`

These two features often show visible group structure and work well for a
two-dimensional DBSCAN demonstration.

### Dataset Characteristics

- Source: loaded directly from the web  
- Total observations: 150  
- Number of features in the original dataset: 4 numeric variables  
- Features used in this notebook: 2 numeric variables  
- Missing values: none expected  

### Why this dataset works for DBSCAN

Although Iris is not as curved or irregular as synthetic moon-shaped data, it
still provides a good real, publicly hosted dataset for demonstrating how
DBSCAN separates dense regions and handles possible boundary points.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rice_ml.unsupervised_learning.dbscan import DBSCAN
from rice_ml.processing.preprocessing import standardize

url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/iris.csv"
df = pd.read_csv(url)

df = df[["Petal.Length", "Petal.Width", "Species"]]
df.head()



## 4. Exploratory Data Analysis (EDA)

For DBSCAN, exploratory analysis is mainly about understanding the **shape,
spacing, and density** of the observations.

Key questions include:

- Do the points form visibly separated groups?
- Are there boundary observations between groups?
- Is standardization needed before clustering?
- What values of `eps` and `min_samples` seem reasonable?

Because DBSCAN depends on distance, feature scaling is an important preprocessing step.


In [ ]:

X = df[["Petal.Length", "Petal.Width"]].copy()

plt.figure(figsize=(8, 6))
plt.scatter(X["Petal.Length"], X["Petal.Width"], alpha=0.8)
plt.xlabel("Petal Length")
plt.ylabel("Petal Width")
plt.title("Raw Iris Data: Petal Length vs Petal Width")
plt.grid(alpha=0.3)
plt.show()



The scatterplot shows that the data has visible group structure, especially one
cluster that appears clearly separated from the others. However, the variables
are measured on different numeric scales, so standardization is appropriate
before applying DBSCAN.


In [ ]:

X_scaled = standardize(X.values)

plt.figure(figsize=(8, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], alpha=0.8)
plt.xlabel("Standardized Petal Length")
plt.ylabel("Standardized Petal Width")
plt.title("Standardized Feature Space")
plt.grid(alpha=0.3)
plt.show()



After standardization, both features contribute more fairly to Euclidean
distance calculations. This is important because DBSCAN determines cluster
membership entirely from neighborhood distances.



## 5. Fit DBSCAN from `rice_ml`

We now fit the from-scratch DBSCAN implementation. The exact choice of
hyperparameters can change the number of clusters and the amount of noise
detected, so these values are selected to give a clear illustration on the
standardized Iris feature space.


In [ ]:

dbscan = DBSCAN(eps=0.35, min_samples=5)
labels = dbscan.fit_predict(X_scaled)

labels[:10]


In [ ]:

unique_labels, counts = np.unique(labels, return_counts=True)
cluster_summary = pd.DataFrame({
    "Cluster Label": unique_labels,
    "Count": counts
})
cluster_summary



In many DBSCAN implementations, the label `-1` represents noise points. Any
nonnegative value corresponds to an identified cluster.


In [ ]:

plt.figure(figsize=(8, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, alpha=0.85)
plt.xlabel("Standardized Petal Length")
plt.ylabel("Standardized Petal Width")
plt.title("DBSCAN Clustering Results")
plt.grid(alpha=0.3)
plt.show()



## 6. Interpretation of the Clusters

The DBSCAN output shows how the algorithm groups observations by density rather
than by forcing all points into a fixed number of clusters.

Possible observations from the result:

- A dense, compact group is usually identified very clearly  
- Some observations near cluster boundaries may be attached to nearby dense regions  
- A few points may be labeled as noise if they are not surrounded by enough neighbors  

This behavior differs from K-Means, which always assigns every point to a cluster.
DBSCAN is more flexible because it allows uncertain or isolated observations to
remain outside the main groups.



## 7. Strengths and Limitations of DBSCAN

### Strengths

- Works well for density-connected groups  
- Can identify outliers naturally  
- Does not require pre-specifying the number of clusters  
- Can recover non-spherical structure better than centroid-based methods  

### Limitations

- The results can be sensitive to the choice of `eps` and `min_samples`  
- Performance may decrease when clusters have very different densities  
- In higher dimensions, distance measures can become less informative  
- Some datasets may require trial and error to find useful hyperparameters



## 8. Conclusion

This notebook demonstrated how to apply DBSCAN using the `rice_ml`
implementation while loading the dataset **directly from an online source**.

Using standardized petal measurements from the Iris dataset, DBSCAN was able to
detect density-based structure without requiring the number of clusters in
advance. This example also shows an important practical point: for distance-based
clustering, careful preprocessing and hyperparameter selection matter just as
much as the clustering algorithm itself.

The main advantage of DBSCAN is its flexibility. It can detect clusters based
on neighborhood connectivity and can separate noise from dense regions, which
makes it a useful alternative to K-Means in many unsupervised learning tasks.
